## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

train = pd.read_pickle('data/train_processed.pkl')
test = pd.read_pickle('data/test_processed.pkl')

!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow

EXPERIMENT_NAME = 'LightGBM_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2ba0a421-1e3f-4270-95f6-f620c6136bf7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d12c9eb2fdb60022d3a1fa42480e50acf473ded525f9f0d8fb56c69586031b1d




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI: https://dagshub.com/skupr23/ml_final.mlflow
Experiment  : LightGBM_Training


## Cell 2 — WMAE Metric

In [2]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 3 - Chronological validation split


In [3]:
train = train.sort_values('Date')
val = train[train['Date'] >= train['Date'].max() - pd.Timedelta(weeks=39)].copy()
fit = train[train['Date'] < val['Date'].min()].copy()
print('fit ends:', fit['Date'].max())
print('val range:', val['Date'].min(), 'to', val['Date'].max())

fit ends: 2012-01-20 00:00:00
val range: 2012-01-27 00:00:00 to 2012-10-26 00:00:00


# Cell 4 - Baseline: 52-week seasonal naive


In [4]:
baseline_pred = val['lag_52'].fillna(val['storedept_median_sales'])
baseline_score = wmae(val['Weekly_Sales'], baseline_pred, val['IsHoliday'])
print('52-week seasonal baseline WMAE:', baseline_score)

52-week seasonal baseline WMAE: 1789.9133525504185


## Cell 5 — Prepare Features (Excluding Leaky Lags)

In [5]:
cat_cols = ['Store','Dept','Type','holiday_type']
for df in [fit, val, test]:
    for c in cat_cols:
        df[c] = df[c].astype('category')

UNAVAILABLE_AT_INFERENCE = ['lag_1', 'lag_2', 'lag_4', 'lag_13', 'lag_26', 'sales_change_1']
drop_cols = ['Weekly_Sales','Date','Id'] if 'Id' in fit.columns else ['Weekly_Sales','Date']
drop_cols += UNAVAILABLE_AT_INFERENCE
feature_cols = [c for c in fit.columns if c not in drop_cols]

X_fit, y_fit = fit[feature_cols], fit['Weekly_Sales']
X_val, y_val = val[feature_cols], val['Weekly_Sales']
print('Total features going in:', len(feature_cols))

Total features going in: 42


## Cell 6 — Train LightGBM on All Features

In [6]:
import lightgbm as lgb

model_all = lgb.LGBMRegressor(
    objective='mae', n_estimators=2000, learning_rate=0.03,
    num_leaves=64, random_state=42
)
model_all.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.062656 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5502
[LightGBM] [Info] Number of data points in the train set: 303030, number of used features: 40
[LightGBM] [Info] Start training from score 7679.899902
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 2166.56
[200]	valid_0's l1: 1482.45
[300]	valid_0's l1: 1441.15
[400]	valid_0's l1: 1424.96
[500]	valid_0's l1: 1406.77
[600]	valid_0's l1: 1396.7
[700]	valid_0's l1: 1391.12
[800]	valid_0's l1: 1385.61
[900]	valid_0's l1: 1383.67
[1000]	valid_0's l1: 1380.23
[1100]	valid_0's l1: 1377.67
[1200]	valid_0's l1: 1375.67
[1300]	valid_0's l1: 1373.94
[1400]	valid_0's l1: 1372.14
[1500]	valid_0's l1: 1371.42
[1600]	valid_0's l1: 1370.07
[1700]	valid_0's l1: 1369.55
[1800]	valid_0's l1: 1368.86
Early stopping, b

LGBMRegressor(learning_rate=0.03, n_estimators=2000, num_leaves=64,
              objective='mae', random_state=42)

## Cell 7 — Evaluate the All-Features Model

In [7]:
val_pred_all = model_all.predict(X_val)
wmae_all = wmae(y_val, val_pred_all, val['IsHoliday'])
print('LightGBM (all features) WMAE:', wmae_all)
print('vs baseline:', baseline_score)

LightGBM (all features) WMAE: 1384.8175014866931
vs baseline: 1789.9133525504185


## Cell 8 — Feature Importance → Feature Selection

In [8]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_all.feature_importances_
}).sort_values('importance', ascending=False)
print(importances)

                    feature  importance
1                      Dept       22461
0                     Store       16631
33           rolling_mean_4        7446
27             week_of_year        6292
41   storedept_median_sales        5854
4                Fuel_Price        4954
3               Temperature        4914
31                   lag_52        4807
28                 week_sin        4221
34         rolling_median_8        3219
29                 week_cos        3177
36          rolling_mean_26        2938
30                   lag_51        2871
35          rolling_mean_13        2860
37     sales_per_store_size        2811
32                   lag_53        2682
10                      CPI        2354
26                    month        2111
39        dept_median_sales        1326
11             Unemployment        1304
7                 MarkDown3        1006
6                 MarkDown2         751
9                 MarkDown5         712
24             holiday_type         591


## Cell 9 — Select Top Features

In [9]:
selected_features = importances[importances['importance'] > 0]['feature'].tolist()
print(f'Dropped {len(feature_cols) - len(selected_features)} zero-importance features')
print('Kept:', len(selected_features))

Dropped 6 zero-importance features
Kept: 36


## Cell 10 — Retrain on Selected Features Only

In [10]:
X_fit_sel = fit[selected_features]
X_val_sel = val[selected_features]

model_sel = lgb.LGBMRegressor(
    objective='mae', n_estimators=2000, learning_rate=0.03,
    num_leaves=64, random_state=42
)
model_sel.fit(
    X_fit_sel, y_fit,
    eval_set=[(X_val_sel, y_val)],
    categorical_feature=[c for c in cat_cols if c in selected_features],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.042008 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5494
[LightGBM] [Info] Number of data points in the train set: 303030, number of used features: 36
[LightGBM] [Info] Start training from score 7679.899902
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 2166.52
[200]	valid_0's l1: 1482.33
[300]	valid_0's l1: 1440.97
[400]	valid_0's l1: 1423.06
[500]	valid_0's l1: 1403.22
[600]	valid_0's l1: 1392.72
[700]	valid_0's l1: 1386.81
[800]	valid_0's l1: 1384.43
[900]	valid_0's l1: 1380.27
[1000]	valid_0's l1: 1375.13
[1100]	valid_0's l1: 1372.2
[1200]	valid_0's l1: 1371.25
[1300]	valid_0's l1: 1369.77
[1400]	valid_0's l1: 1367.98
Early stopping, best iteration is:
[1435]	valid_0's l1: 1367.52


LGBMRegressor(learning_rate=0.03, n_estimators=2000, num_leaves=64,
              objective='mae', random_state=42)

## Cell 11 — Compare: Did Feature Selection Help?

In [11]:
val_pred_sel = model_sel.predict(X_val_sel)
wmae_sel = wmae(y_val, val_pred_sel, val['IsHoliday'])

print('Baseline (52wk):        ', baseline_score)
print('LightGBM all features:  ', wmae_all)
print('LightGBM selected feats:', wmae_sel)

best_model = model_sel if wmae_sel <= wmae_all else model_all
best_wmae = min(wmae_sel, wmae_all)
print('\nBest LightGBM variant WMAE:', best_wmae)

Baseline (52wk):         1789.9133525504185
LightGBM all features:   1384.8175014866931
LightGBM selected feats: 1384.9567519479626

Best LightGBM variant WMAE: 1384.8175014866931


## Cell 12 — Holiday vs Non-Holiday Breakdown

In [12]:
best_pred = val_pred_sel if best_model is model_sel else val_pred_all
is_hol = val['IsHoliday'].values

print('Holiday WMAE:    ', wmae(y_val[is_hol], best_pred[is_hol], val['IsHoliday'][is_hol]))
print('Non-holiday WMAE:', wmae(y_val[~is_hol], best_pred[~is_hol], val['IsHoliday'][~is_hol]))

Holiday WMAE:     1464.6183572588563
Non-holiday WMAE: 1363.6680381876565


## Cell 13 — Save Model Locally

In [13]:
import os, joblib
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/lightgbm_best.pkl')
print('Saved.')

Saved.


## Cell 13b — Retrain Final Model on Full Labeled Data

In [14]:
final_features_used = selected_features if best_model is model_sel else feature_cols

for c in cat_cols:
    train[c] = train[c].astype('category')

X_full_sel = train[final_features_used]
y_full = train['Weekly_Sales']

best_params = best_model.get_params()
best_n_estimators = best_model.best_iteration_
final_n_estimators = int(best_n_estimators * 1.1)
best_params_clean = {k: v for k, v in best_params.items() if k != 'n_estimators'}

model_final_submission = lgb.LGBMRegressor(**best_params_clean, n_estimators=final_n_estimators)
model_final_submission.fit(
    X_full_sel, y_full,
    categorical_feature=[c for c in cat_cols if c in final_features_used]
)
print('Retrained on full labeled data:', len(X_full_sel), 'rows')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066740 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5568
[LightGBM] [Info] Number of data points in the train set: 421570, number of used features: 40
[LightGBM] [Info] Start training from score 7612.029785
Retrained on full labeled data: 421570 rows


---
# Production pipeline

The requirement is that the saved pipeline runs on the **raw, un-preprocessed test
set**. The model above was trained on `data/*_processed.pkl`, so the whole of
`feature_engineering.ipynb` has to be reachable from inside the pipeline object.

`WalmartFeatureEngineer` below is that notebook re-expressed as a scikit-learn
transformer. It changes nothing about the features: it learns the train history,
rolling tails and median aggregates in `fit`, and rebuilds the same 53 columns in
`transform`. The final pipeline is

    raw rows -> WalmartFeatureEngineer -> CategoricalCaster -> FeatureSelector -> model

## Cell 14 — Raw-Input Pipeline Components (WalmartFeatureEngineer, CategoricalCaster, FeatureSelector)

In [15]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
LAG_WEEKS = [1, 2, 4, 13, 26, 51, 52, 53]
ROLL_COLS = ['rolling_mean_4', 'rolling_median_8', 'rolling_mean_13', 'rolling_mean_26']

SUPERBOWL = ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']
LABORDAY = ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']
THANKSGIVING = ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']
CHRISTMAS = ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']


def holiday_type_of(date):
    d = date.strftime('%Y-%m-%d')
    if d in SUPERBOWL: return 'SuperBowl'
    if d in LABORDAY: return 'LaborDay'
    if d in THANKSGIVING: return 'Thanksgiving'
    if d in CHRISTMAS: return 'Christmas'
    return 'Normal'


class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    """feature_engineering.ipynb, expressed as a transformer."""

    def __init__(self, features, stores, lag_weeks=tuple(LAG_WEEKS)):
        self.features = features
        self.stores = stores
        self.lag_weeks = lag_weeks

    def fit(self, X, y=None):
        raw = X.copy()
        raw['Date'] = pd.to_datetime(raw['Date'])
        if y is not None:
            raw['Weekly_Sales'] = np.asarray(y)
        raw = raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        self.history_ = raw[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        self.train_keys_ = raw[['Store', 'Date']].drop_duplicates()

        rs = self.history_.copy()
        g = rs.groupby(['Store', 'Dept'])['Weekly_Sales']
        rs['rolling_mean_4'] = g.shift(1).rolling(4).mean().reset_index(drop=True)
        rs['rolling_median_8'] = g.shift(1).rolling(8).median().reset_index(drop=True)
        rs['rolling_mean_13'] = g.shift(1).rolling(13).mean().reset_index(drop=True)
        rs['rolling_mean_26'] = g.shift(1).rolling(26).mean().reset_index(drop=True)

        self.last_roll_ = (rs.sort_values('Date')
                             .groupby(['Store', 'Dept'])[ROLL_COLS]
                             .last().reset_index())

        self.dept_median_ = raw.groupby('Dept')['Weekly_Sales'].median().rename('dept_median_sales')
        self.store_median_ = raw.groupby('Store')['Weekly_Sales'].median().rename('store_median_sales')
        self.storedept_median_ = (raw.groupby(['Store', 'Dept'])['Weekly_Sales']
                                     .median().rename('storedept_median_sales'))
        return self

    def _macro(self, test_keys):
        combined = pd.concat([self.train_keys_, test_keys], ignore_index=True).drop_duplicates()
        combined = combined.merge(self.features[['Store', 'Date', 'CPI', 'Unemployment']],
                                  on=['Store', 'Date'], how='left')
        combined = combined.sort_values(['Store', 'Date'])
        for col in ['CPI', 'Unemployment']:
            combined[f'{col}_missing'] = combined[col].isna().astype('int8')
            combined[col] = combined.groupby('Store')[col].ffill()
        return combined

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        df = (df.merge(self.features, on=['Store', 'Date', 'IsHoliday'],
                       how='left', validate='many_to_one')
                .merge(self.stores, on='Store', how='left', validate='many_to_one'))

        for col in MARKDOWN_COLS:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
            df[col] = df[col].fillna(0)
        df['markdown_total'] = df[MARKDOWN_COLS].sum(axis=1)
        df['markdown_count'] = (df[MARKDOWN_COLS] > 0).sum(axis=1)
        df['has_any_markdown'] = (df['markdown_total'] > 0).astype('int8')

        macro = self._macro(df[['Store', 'Date']].drop_duplicates())
        df = df.drop(columns=['CPI', 'Unemployment']).merge(macro, on=['Store', 'Date'], how='left')

        df['holiday_type'] = df['Date'].apply(holiday_type_of)
        df['year'] = df['Date'].dt.year
        df['month'] = df['Date'].dt.month
        df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

        for w in self.lag_weeks:
            shifted = self.history_.copy()
            shifted['Date'] = shifted['Date'] + pd.Timedelta(weeks=w)
            shifted = shifted.rename(columns={'Weekly_Sales': f'lag_{w}'})
            df = df.merge(shifted, on=['Store', 'Dept', 'Date'], how='left')

        df = df.merge(self.last_roll_, on=['Store', 'Dept'], how='left')
        df['sales_change_1'] = df['lag_1'] - df['lag_2']
        df['sales_per_store_size'] = df['lag_52'] / df['Size']
        df['markdown_per_store_size'] = df['markdown_total'] / df['Size']

        df = (df.merge(self.dept_median_, on='Dept', how='left')
                .merge(self.store_median_, on='Store', how='left')
                .merge(self.storedept_median_, on=['Store', 'Dept'], how='left'))
        return df


class CategoricalCaster(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype('category')
        return X


class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.features]


def build_full_pipeline(fitted_model, model_features, feature_engineer):
    return Pipeline([
        ('feature_engineering', feature_engineer),
        ('cast_categoricals', CategoricalCaster(cat_cols)),
        ('select_features', FeatureSelector(model_features)),
        ('model', fitted_model),
    ])

## Cell 15 — Fit Feature Engineer on Raw Training Data

In [16]:
train_raw = pd.read_csv('data/train.csv', parse_dates=['Date'])
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
features_raw = pd.read_csv('data/features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv('data/stores.csv')

feature_engineer = WalmartFeatureEngineer(features_raw, stores_raw).fit(
    train_raw.drop(columns='Weekly_Sales'),
    train_raw['Weekly_Sales'],
)
print('Feature engineer fitted on', len(train_raw), 'raw training rows.')

Feature engineer fitted on 421570 raw training rows.


## Cell 16 — Build Final Pipeline (Full-Data Model)

In [17]:
best_features = final_features_used
best_variant = 'all_features' if best_model is model_all else 'selected_features'
final_pipeline = build_full_pipeline(model_final_submission, best_features, feature_engineer)
print('Best variant:', best_variant, '|', len(best_features), 'features (full-data retrain)')

Best variant: all_features | 42 features (full-data retrain)


## Cell 17 — Sanity Check Against the Full-Data Model

In [18]:
_ref = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
_ref_pred = model_final_submission.predict(_ref[best_features])
_pipe_pred = final_pipeline.predict(test_raw)

pipeline_max_abs_diff = float(np.abs(_ref_pred - _pipe_pred).max())
print('max |pipeline(raw) - model(processed)| =', pipeline_max_abs_diff)
assert pipeline_max_abs_diff < 1e-6, 'pipeline does not reproduce the trained model'

max |pipeline(raw) - model(processed)| = 0.0


## Cell 18 — MLflow: Cleaning, Baseline, Feature Selection, Validation Runs

In [19]:
import os, tempfile
import mlflow.sklearn

PREPROCESSING_STEPS = [
    {'step': 'load_raw', 'detail': 'read train/test/features/stores, parse Date, sort by Store-Dept-Date'},
    {'step': 'merge', 'detail': "features on ['Store','Date','IsHoliday'] + stores on ['Store']"},
    {'step': 'markdown_missing_indicators', 'detail': 'MarkDown*_missing flags, fillna(0), markdown_total, markdown_count, has_any_markdown'},
    {'step': 'macro_forward_fill', 'detail': 'per-Store ffill of CPI / Unemployment across train+test'},
    {'step': 'holiday_type', 'detail': 'SuperBowl / LaborDay / Thanksgiving / Christmas / Normal'},
    {'step': 'calendar_features', 'detail': 'year, month, week_of_year, week_sin, week_cos'},
    {'step': 'exact_date_lags', 'detail': 'date-shifted merges; test looks back into train history only'},
    {'step': 'rolling_statistics', 'detail': 'mean_4 / median_8 / mean_13 / mean_26; test gets the last train value'},
    {'step': 'trend_and_size_features', 'detail': 'sales_change_1, sales_per_store_size, markdown_per_store_size'},
    {'step': 'fallback_aggregates', 'detail': 'dept / store / store-dept median sales, from train only'},
]

with mlflow.start_run(run_name='LightGBM_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'markdown_cols': MARKDOWN_COLS, 'lag_weeks': LAG_WEEKS,
        'rolling_windows': [4, 8, 13, 26], 'forward_filled_cols': ['CPI', 'Unemployment'],
        'merge_validate': 'many_to_one', 'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0], 'train_cols': train.shape[1],
        'test_rows': test.shape[0], 'test_cols': test.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()), 'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

with mlflow.start_run(run_name='LightGBM_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_param('strategy', 'lag_52, falling back to storedept_median_sales')
    mlflow.log_metric('val_wmae', baseline_score)

with mlflow.start_run(run_name='LightGBM_Feature_Selection'):
    mlflow.set_tag('stage', 'feature_selection')
    mlflow.log_params(model_all.get_params())
    mlflow.log_params({
        'selection_rule': 'importance > 0 on the all-features model',
        'n_features_before': len(feature_cols), 'n_features_after': len(selected_features),
    })
    mlflow.log_metrics({
        'val_wmae_all_features': wmae_all, 'val_wmae_selected_features': wmae_sel,
        'n_features_dropped': len(feature_cols) - len(selected_features),
        'best_iteration': model_all.best_iteration_,
    })
    mlflow.log_dict({'features': feature_cols}, 'features/all_features.json')
    mlflow.log_dict({'features': selected_features}, 'features/selected_features.json')
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'feature_importances.csv')
        importances.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='features')

with mlflow.start_run(run_name='LightGBM_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series)',
        'val_weeks': 39, 'fit_end': str(fit['Date'].max().date()),
        'val_start': str(val['Date'].min().date()), 'val_end': str(val['Date'].max().date()),
        'fit_rows': len(fit), 'val_rows': len(val),
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score, 'wmae_all_features': wmae_all,
        'wmae_selected_features': wmae_sel, 'best_wmae': best_wmae,
        'best_wmae_holiday': wmae(y_val[is_hol], best_pred[is_hol], val['IsHoliday'][is_hol]),
        'best_wmae_non_holiday': wmae(y_val[~is_hol], best_pred[~is_hol], val['IsHoliday'][~is_hol]),
        'improvement_over_baseline': baseline_score - best_wmae,
    })

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run LightGBM_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4/runs/b96fcae5e06a4ef0a7b29eca3a47c8c0
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4
🏃 View run LightGBM_Baseline at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4/runs/fd243871962c4c568554703f1b986911
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4
🏃 View run LightGBM_Feature_Selection at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4/runs/fb2ba8b91d274a26a54b95ceb60256b7
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4
🏃 View run LightGBM_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4/runs/b20c523964b74e56beea7fc1cf4841f8
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4


## Cell 19 — Final Model + Raw-Input Pipeline

In [20]:
with mlflow.start_run(run_name='LightGBM_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'lightgbm', 'variant': best_variant})
    mlflow.log_params(best_model.get_params())
    mlflow.log_param('n_features', len(best_features))
    mlflow.log_metrics({
        'final_wmae': best_wmae, 'baseline_wmae': baseline_score,
        'pipeline_vs_processed_max_abs_diff': pipeline_max_abs_diff,
    })

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='pipeline',
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        input_example=test_raw.head(5),
    )
    mlflow.log_artifact('models/lightgbm_best.pkl', artifact_path='estimator')

    FINAL_RUN_ID = final_run.info.run_id
    FINAL_MODEL_URI = f'runs:/{FINAL_RUN_ID}/pipeline'

print('LightGBM final run:', FINAL_RUN_ID)

2026/07/12 12:44:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 12:44:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

🏃 View run LightGBM_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4/runs/0f7516616f514c7bb51674d252e48be6
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/4
LightGBM final run: 0f7516616f514c7bb51674d252e48be6


## Cell 20 — Register (Only If LightGBM Wins Overall)

In [21]:
REGISTER_AS_GLOBAL_BEST = False

if REGISTER_AS_GLOBAL_BEST:
    registered_version = mlflow.register_model(
        model_uri=FINAL_MODEL_URI,
        name=REGISTERED_MODEL_NAME,
    )
    print('Registered model:', REGISTERED_MODEL_NAME, '| version:', registered_version.version)
else:
    print('Pipeline logged and saved, not registered. Register only if LightGBM wins overall.')

Pipeline logged and saved, not registered. Register only if LightGBM wins overall.


## Generate Submission CSV

In [22]:
test_predictions = final_pipeline.predict(test_raw)

test_ids = (
    test_raw['Store'].astype(str) + '_' +
    test_raw['Dept'].astype(str) + '_' +
    test_raw['Date'].dt.strftime('%Y-%m-%d')
)

submission = pd.DataFrame({
    'Id': test_ids,
    'Weekly_Sales': test_predictions
})

submission.to_csv('submission.csv', index=False)
print('Submission CSV created successfully.')

Submission CSV created successfully.
